In [1]:
import sys , os 
import geopandas as gpd 
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
data_path = os.path.join(os.getcwd(),'..','data','geojson','polygon.geojson')
data = gpd.read_file(data_path)

In [2]:
data

,CODIGO,OBSERV,INSUMO,APOYO,ASIGNACION,Asignado,OAM,area,geometry
0,231,clean pasture,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,34429.374,"MULTIPOLYGON (((-72.97056 10.61903, -72.97054 ..."
1,21,seasonal crops,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,6784.769,"MULTIPOLYGON (((-72.96914 10.61884, -72.96915 ..."
2,112,urban not continuous,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,12227.436,"MULTIPOLYGON (((-72.96928 10.61948, -72.96931 ..."
3,231,clean pasture,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,19385.187,"MULTIPOLYGON (((-72.97264 10.61976, -72.97273 ..."
4,313,fragmented forest,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,5013.914,"MULTIPOLYGON (((-72.96917 10.62053, -72.9691 1..."
...,...,...,...,...,...,...,...,...,...
1950,232,wooded pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,10536.989,"MULTIPOLYGON (((-75.12278 3.8104, -75.12282 3...."
1951,231,clean pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,5139.198,"MULTIPOLYGON (((-75.12367 3.81079, -75.12357 3..."
1952,231,clean pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,4458.659,"MULTIPOLYGON (((-75.12605 3.81572, -75.12607 3..."
1953,None,None,None,None,None,None,None,NaN,None


In [3]:
from src.preprocess import build_image_download_uri 

In [4]:
gdf = await build_image_download_uri(data,'OAM')

Processing imagery links: 100%|██████████| 1955/1955 [11:26<00:00,  2.85it/s]


In [5]:
gdf['poly_id'] = range(len(gdf))

In [7]:
gdf = gdf[gdf['OAM'].notna() & (gdf['OAM'] != '')] # drop null rows without images 

In [9]:
len(gdf)

1953

In [10]:
from src.preprocess import assign_image_uids 

In [11]:
gdf, _ = assign_image_uids(gdf, 'download_url')

/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/.venv/lib/python3.13/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [12]:
print( gdf['image_uid'].nunique(), gdf['download_url'].nunique() ) 

30 30


In [28]:
gdrive_df = gdf[gdf['url_type'] == 'gdrive'][['image_uid', 'download_url']].drop_duplicates()
gdrive_gdf = dict(zip(gdrive_df['image_uid'], gdrive_df['download_url']))

In [29]:
gdrive_gdf

{'img_0004_befb8cbc': 'https://drive.google.com/drive/folders/1QRkuTFi_F-_uvWXQMtEvoUKSD_T12-7F?usp=drive_link',
 'img_0005_b3f008c4': 'https://drive.google.com/drive/folders/1LXcT0lrf1_MqfedpQHI6tf_qBJxd03VQ?usp=drive_link',
 'img_0014_0aad0f9c': 'https://drive.google.com/drive/folders/1PoaKqUvMOJI90EOKiKBlT5OXgkUYh-kQ?usp=sharing',
 'img_0015_0f954bee': 'https://drive.google.com/drive/folders/19TyWiBf5cd0-OTJbFz0G5qLoQL3Xga-0?usp=drive_link',
 'img_0016_8cb9e2ca': 'https://drive.google.com/drive/folders/1tMNj7BR3uq20TUqxDsnZQihXJRPNakxJ?usp=drive_link',
 'img_0017_f8406eda': 'https://drive.google.com/drive/folders/14rikAgD3b-cd_lmgeXK7qmHKB62qY0sv?usp=drive_link',
 'img_0018_3c5c5a9c': 'https://drive.google.com/drive/folders/1e1e--U2iTxCx-of2m2HWiGEAnMUivtMm?usp=drive_link'}

In [ ]:
gdrive_gdf_fixed = gdrive_gdf # lets fix those group bastards from google 
gdrive_gdf_fixed['img_0004_befb8cbc'] = 'https://drive.google.com/file/d/1uDznoDuGo5OsSnSd3JFxOvYB0rVuXCIT/view?usp=sharing'
gdrive_gdf_fixed['img_0005_b3f008c4'] = 'https://drive.google.com/file/d/1KrC1AXL5IcmGCj_havxxlzG_gnGDzjAv/view?usp=sharing'
gdrive_gdf_fixed['img_0014_0aad0f9c'] = 'https://drive.google.com/file/d/1T1Pin5cXsCcu42Ogx1apB-_2avYBwsv8/view?usp=sharing'
gdrive_gdf_fixed['img_0015_0f954bee'] = 'https://drive.google.com/file/d/1Pz9em-zdp8yNXEKZPUrtiFNJEl3MUq_8/view?usp=sharing'
gdrive_gdf_fixed['img_0016_8cb9e2ca'] = 'https://drive.google.com/file/d/1zPcUXSBDHLsMpPslu4VcT29XiIb2sB0J/view?usp=sharing'
gdrive_gdf_fixed['img_0017_f8406eda'] = 'https://drive.google.com/file/d/1skz0zSyMGh6R5WC8MVe2kzfPr4DGyc6O/view?usp=sharing'
gdrive_gdf_fixed['img_0018_3c5c5a9c'] = 'https://drive.google.com/file/d/1CqNSAEJhfVvpryHvwEy43HJi0dyn_94C/view?usp=sharing'


In [31]:
gdrive_gdf_fixed

{'img_0004_befb8cbc': 'https://drive.google.com/file/d/1uDznoDuGo5OsSnSd3JFxOvYB0rVuXCIT/view?usp=sharing',
 'img_0005_b3f008c4': 'https://drive.google.com/file/d/1KrC1AXL5IcmGCj_havxxlzG_gnGDzjAv/view?usp=sharing',
 'img_0014_0aad0f9c': 'https://drive.google.com/file/d/1T1Pin5cXsCcu42Ogx1apB-_2avYBwsv8/view?usp=sharing',
 'img_0015_0f954bee': 'https://drive.google.com/file/d/1Pz9em-zdp8yNXEKZPUrtiFNJEl3MUq_8/view?usp=sharing',
 'img_0016_8cb9e2ca': 'https://drive.google.com/file/d/1zPcUXSBDHLsMpPslu4VcT29XiIb2sB0J/view?usp=sharing',
 'img_0017_f8406eda': 'https://drive.google.com/file/d/1skz0zSyMGh6R5WC8MVe2kzfPr4DGyc6O/view?usp=sharing',
 'img_0018_3c5c5a9c': 'https://drive.google.com/file/d/1CqNSAEJhfVvpryHvwEy43HJi0dyn_94C/view?usp=sharing'}

In [32]:
gdf['download_url'] = gdf['image_uid'].map(gdrive_gdf_fixed).fillna(gdf['download_url'])

/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/.venv/lib/python3.13/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [34]:
gdf.to_file(os.path.join(os.getcwd(),'..','data','geojson','polygon_cleaned.geojson'),driver='GeoJSON')

In [35]:
from src.preprocess import download_images

In [36]:
await download_images(gdf,os.path.join(os.getcwd(),'..','data','images'))

From (original): https://drive.google.com/uc?id=1zPcUXSBDHLsMpPslu4VcT29XiIb2sB0J
From (redirected): https://drive.google.com/uc?id=1zPcUXSBDHLsMpPslu4VcT29XiIb2sB0J&confirm=t&uuid=007f455a-003a-4c88-a7b1-f2e24a9a2d87
To: /home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/data/images/img_0016_8cb9e2ca.tif
Downloading...
From (original): https://drive.google.com/uc?id=1zPcUXSBDHLsMpPslu4VcT29XiIb2sB0J
From (redirected): https://drive.google.com/uc?id=1zPcUXSBDHLsMpPslu4VcT29XiIb2sB0J&confirm=t&uuid=007f455a-003a-4c88-a7b1-f2e24a9a2d87
To: /home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/data/images/img_0016_8cb9e2ca.tif
Downloading...
From (original): https://drive.google.com/uc?id=1uDznoDuGo5OsSnSd3JFxOvYB0rVuXCIT
From (redirected): https://drive.google.com/uc?id=1uDznoDuGo5OsSnSd3JFxOvYB0rVuXCIT&confirm=t&uuid=8bef26c9-5950-4bda-9a92-5a3f9b465930
To: /home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/data/images/img_0004_befb8cbc.tif


Download

['/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/notebooks/../data/images/img_0011_7e70dcb2.tif',
 '/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/notebooks/../data/images/img_0000_c4a77f3e.tif',
 '/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/notebooks/../data/images/img_0024_95971f64.tif',
 '/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/notebooks/../data/images/img_0016_8cb9e2ca.tif',
 '/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/notebooks/../data/images/img_0004_befb8cbc.tif',
 '/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/notebooks/../data/images/img_0015_0f954bee.tif',
 '/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/notebooks/../data/images/img_0002_07ccbe0e.tif',
 '/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/notebooks/../data/images/img_0017_f8406eda.tif',
 '/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/notebooks/../d

Check if images are COG or not , if not we need to make them cog 

In [37]:
bash_script_path = os.path.join(os.getcwd(),'..','tif2cog.sh')

In [38]:
! bash $bash_script_path ../data/images

1/30 Converting: img_0026_40a8e8df.tif
2/30 Converting: img_0029_37df798b.tif
3/30 Converting: img_0027_cbc71305.tif
4/30 Converting: img_0023_d68997fa.tif
5/30 Converting: img_0020_4a8f0af8.tif
6/30 Converting: img_0000_c4a77f3e.tif
7/30 Converting: img_0008_4f1122ba.tif
8/30 Converting: img_0028_47d0c657.tif
9/30 Converting: img_0024_95971f64.tif
10/30 Converting: img_0001_f4dde9b0.tif
11/30 Converting: img_0017_f8406eda.tif
12/30 Converting: img_0013_08ee5eac.tif
13/30 Converting: img_0018_3c5c5a9c.tif
14/30 Converting: img_0006_2828f6be.tif
15/30 Converting: img_0009_c57f932c.tif
16/30 Converting: img_0012_f4d05bf8.tif
17/30 Converting: img_0019_2d8e19cb.tif
18/30 Converting: img_0014_0aad0f9c.tif
19/30 Converting: img_0016_8cb9e2ca.tif
20/30 Converting: img_0022_4e02d22a.tif
21/30 Converting: img_0025_4090d330.tif
22/30 Converting: img_0007_eec6a1e9.tif
23/30 Converting: img_0003_b06e8b27.tif
24/30 Converting: img_0005_b3f008c4.tif
25/30 Converting: img_0004_befb8cbc.tif
26/30 Con

In [39]:
image_uid_col='image_uid'
# base_url='https://files.krschap.tech/api/public/dl/RGNv3CL4'

base_url=os.path.join(os.getcwd(),'..','data','images','cog')
row = gdf.iloc[0]
row

CODIGO                                                        231
OBSERV                                              clean pasture
INSUMO                                      Villanueva-orthophoto
APOYO                                                        None
ASIGNACION                                                  Sofia
Asignado                                                     None
OAM             https://map.openaerialmap.org/#/-72.9698181152...
area                                                    34429.374
geometry        MULTIPOLYGON (((-72.97055585411157 10.61902892...
url_type                                                      oam
download_url    https://oin-hotosm-temp.s3.us-east-1.amazonaws...
poly_id                                                         0
image_uid                                       img_0000_c4a77f3e
Name: 0, dtype: object

In [40]:
cog_url = f"{base_url}/{row[image_uid_col]}.tif"
geom = [row.geometry.__geo_interface__]
cog_url

'/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/notebooks/../data/images/cog/img_0000_c4a77f3e.tif'

In [ ]:
# import requests

# response = requests.get(cog_url, headers={'Range': 'bytes=0-1023'})
# print(f"Status: {response.status_code}")
# print(f"Content-Length: {len(response.content)}")
# print(f"Should be ~1024 bytes, got: {len(response.content)}")

In [41]:
import rasterio 
from rasterio.mask import mask
import numpy as np 
with rasterio.open(cog_url) as src:
    print("Raster CRS:", src.crs)
    print("Raster bounds:", src.bounds)
    
    print("\nOriginal geometry CRS:", data.crs)
    print("Original geometry bounds:", row.geometry.bounds)
    
    geom_gdf = gpd.GeoDataFrame([row], geometry='geometry', crs=data.crs)
    geom_reprojected = geom_gdf.to_crs(src.crs)
    geom_transformed = [geom_reprojected.geometry.iloc[0].__geo_interface__]
    
    print("\nTransformed bounds:", geom_reprojected.geometry.iloc[0].bounds)
    
    masked_data, _ = mask(src, geom_transformed, crop=True, all_touched=False)

    
    masked_data, _ = mask(src, geom_transformed, crop=True, all_touched=False)

    stats = {}
    for band_idx, band_name in enumerate(['r', 'g', 'b'], start=1):
        band_data = masked_data[band_idx - 1]
        valid = band_data[band_data != src.nodata] if not np.ma.isMaskedArray(band_data) else band_data[~band_data.mask]
        
        if valid.size > 0:
            stats[f'{band_name}_mean'] = float(np.mean(valid))
            stats[f'{band_name}_std'] = float(np.std(valid))
        else:
            stats[f'{band_name}_mean'] = np.nan
            stats[f'{band_name}_std'] = np.nan
    print(stats)

Raster CRS: EPSG:32618
Raster bounds: BoundingBox(left=721782.2054845318, bottom=1174560.63571348, right=722310.3216997313, top=1174796.2209395552)

Original geometry CRS: EPSG:4326
Original geometry bounds: (-72.9714814422654, 10.618840012053344, -72.96914035652021, 10.620975138959965)

Transformed bounds: (721925.7615761297, 1174560.7526012938, 722182.7255575805, 1174796.086679114)
{'r_mean': 67.11867219203522, 'r_std': 63.05077827901257, 'g_mean': 72.36945632182152, 'g_std': 65.61500049739803, 'b_mean': 50.0079449993589, 'b_std': 48.51453627072589}
